# Phase 2: Data Cleaning and Preprocessing Pipeline
**Project:** Customer Churn Prediction & Behavioral Segmentation  
**Objective:** Handle missing values, whitespace strings, datatype corrections, binary encoding, and build leak-free Scikit-Learn transformers.


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

DATA_PATH = os.path.join('..', 'data', 'raw', 'Telco-Customer-Churn.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'raw', 'Telco-Customer-Churn.csv')

df_raw = pd.read_csv(DATA_PATH)
print(f'Raw dataset shape: {df_raw.shape}')


## 1. Domain-Specific Data Cleaning
1. Handle whitespace in TotalCharges (11 rows where tenure == 0).
2. Impute with 0.0 because these are newly enrolled customers who have not received their first monthly bill.
3. Strip trailing/leading whitespace from all text columns.


In [ ]:
df_clean = df_raw.copy()

# Fix TotalCharges
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'].astype(str).str.strip(), errors='coerce')
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0.0)

# Strip whitespace
obj_cols = df_clean.select_dtypes(include=['object']).columns
for col in obj_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

print(f'Missing values after cleaning: {df_clean.isnull().sum().sum()}')
print(f'TotalCharges dtype: {df_clean["TotalCharges"].dtype}')
print(f'TotalCharges summary:
{df_clean["TotalCharges"].describe().round(2)}')


## 2. Save Cleaned Dataset for Subsequent Phases
We save this human-readable cleaned dataset to data/processed/telco_cleaned.csv to be used for Exploratory Data Analysis (Phase 3) and RFM Segmentation (Phase 5).


In [ ]:
PROCESSED_PATH = os.path.join('..', 'data', 'processed', 'telco_cleaned.csv')
if not os.path.exists(os.path.dirname(PROCESSED_PATH)):
    PROCESSED_PATH = os.path.join('data', 'processed', 'telco_cleaned.csv')
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)

df_clean.to_csv(PROCESSED_PATH, index=False)
print(f'Successfully saved cleaned dataset to: {PROCESSED_PATH}')


## 3. Scikit-Learn Leakage-Free Preprocessing Pipeline Demonstration
We create a reusable ColumnTransformer that can be fitted exclusively on training splits during modeling.


In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in df_clean.columns if c not in num_cols + ['customerID', 'Churn']]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ]
)

# Test transformation
X_sample = df_clean.drop(columns=['customerID', 'Churn'])
X_transformed = preprocessor.fit_transform(X_sample)
feature_names = preprocessor.get_feature_names_out()

print(f'Transformed feature matrix shape: {X_transformed.shape}')
print(f'Total engineered/encoded features: {len(feature_names)}')
print(f'Sample feature names: {feature_names[:8]}')
